# 🔴 CALDERA RED TEAM PRACTICE LAB
## Laboratorio Avanzado de Red Team con MITRE Caldera

---

**Objetivos de la práctica:**
- Comprender la arquitectura y componentes de MITRE Caldera
- Diseñar y ejecutar operaciones de Red Team con múltiples tácticas ATT&CK
- Crear abilities y adversarios personalizados
- Analizar resultados con Python/Pandas y visualizarlos con Matplotlib/Seaborn
- Correlacionar técnicas ofensivas con capacidades defensivas (Suricata)

**Herramientas:**
| Herramienta | URL | Credenciales |
|-------------|-----|--------------|
| Caldera Web | http://localhost:8888 | admin / admin |
| Jupyter     | http://localhost:8889 | (sin contraseña) |

> ⚠️ **Aviso legal:** Este laboratorio es exclusivamente para fines educativos en un entorno controlado.
---

## Sección 0: Validación del Entorno 🔍

Antes de comenzar verificamos que el servicio de Caldera esté activo y que la API responda correctamente.

In [ ]:
# ── Importaciones globales ──────────────────────────────────────────────────
import subprocess
import requests
import json
import os
import time
import uuid
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# ── Constantes de configuración ─────────────────────────────────────────────
CALDERA_URL = 'http://localhost:8888'
CALDERA_KEY = 'REDADMIN123'
HEADERS     = {'KEY': CALDERA_KEY, 'Content-Type': 'application/json'}

print('✅ Importaciones completadas')
print(f'   Caldera URL : {CALDERA_URL}')
print(f'   Seaborn     : {"disponible" if HAS_SEABORN else "no instalado (pip install seaborn)"}')

In [ ]:
# ── Verificación de servicios systemd ───────────────────────────────────────
def check_service(name):
    result = subprocess.run(['systemctl', 'is-active', name],
                            capture_output=True, text=True)
    status = result.stdout.strip()
    icon   = '✅' if status == 'active' else '❌'
    print(f'{icon} Servicio {name:12s}: {status}')
    return status == 'active'

print('='*55)
print('  CALDERA RED TEAM LAB – Validación del Entorno')
print('='*55)
print('\n── Servicios systemd ──')
ok_caldera = check_service('caldera')
ok_jupyter = check_service('jupyter')

In [ ]:
# ── Verificar conectividad con la API de Caldera ────────────────────────────
def check_url(label, url, headers=None):
    try:
        r = requests.get(url, headers=headers or {}, timeout=5)
        icon = '✅' if r.status_code < 400 else '⚠️'
        print(f'{icon} {label:20s}: HTTP {r.status_code}')
        return r.status_code < 400, r
    except Exception as e:
        print(f'❌ {label:20s}: {e}')
        return False, None

print('\n── Endpoints HTTP ──')
ok_api, r_health = check_url('Caldera API Health', f'{CALDERA_URL}/api/v2/health', HEADERS)

if ok_api and r_health is not None:
    try:
        health = r_health.json()
        print(f'\n📋 Respuesta de /health:')
        print(json.dumps(health, indent=2))
    except Exception:
        print(r_health.text[:300])

In [ ]:
# ── Validar credenciales y mostrar agentes disponibles ──────────────────────
try:
    r_agents = requests.get(f'{CALDERA_URL}/api/v2/agents', headers=HEADERS, timeout=10)
    agents_list = r_agents.json() if r_agents.status_code == 200 else []
    print(f'✅ Credenciales válidas. Agentes registrados: {len(agents_list)}')
    if agents_list:
        print('\n── Detalle de Agentes ──')
        for ag in agents_list:
            paw      = ag.get('paw', 'N/A')
            group    = ag.get('group', 'N/A')
            hostname = ag.get('host', 'N/A')
            platform = ag.get('platform', 'N/A')
            status   = ag.get('trusted', False)
            icon     = '🤖'
            print(f'  {icon} PAW:{paw}  grupo:{group}  host:{hostname}  plataforma:{platform}  confiable:{status}')
    else:
        print('⚠️  No hay agentes registrados. Despliega un agente Sandcat primero.')
except Exception as e:
    print(f'❌ Error al consultar agentes: {e}')

In [ ]:
# ── Mostrar versión de Caldera ───────────────────────────────────────────────
try:
    r_ver = requests.get(f'{CALDERA_URL}/api/v2/health', headers=HEADERS, timeout=5)
    if r_ver.status_code == 200:
        data = r_ver.json()
        version = data.get('version', 'N/A')
        plugins = data.get('plugins', [])
        print(f'�� Caldera versión : {version}')
        print(f'🔌 Plugins activos : {plugins}')
    else:
        print(f'⚠️ No se pudo obtener la versión: HTTP {r_ver.status_code}')
except Exception as e:
    print(f'❌ Error: {e}')

## Sección 1: Introducción Conceptual 📚

### ¿Qué es MITRE Caldera?

**MITRE Caldera** es una plataforma de emulación de adversarios de código abierto desarrollada por MITRE Corporation.
Implementa el framework MITRE ATT&CK para automatizar operaciones de Red Team y evaluaciones de seguridad.

#### Arquitectura
```
┌──────────────────────────────────────────────────────┐
│                  Caldera Server                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────────────┐  │
│  │ REST API │  │ Web GUI  │  │  Plugin Engine   │  │
│  └──────────┘  └──────────┘  └──────────────────┘  │
│  ┌──────────────────────────────────────────────┐   │
│  │            Planning Engine                   │   │
│  │  Sequential | Batch | Landmine | Likely      │   │
│  └──────────────────────────────────────────────┘   │
└─────────────────────┬────────────────────────────────┘
                      │ C2 (HTTP/DNS/SMTP)
         ┌────────────┴────────────┐
    ┌────┴────┐              ┌────┴────┐
    │ Agent 1 │              │ Agent 2 │
    │ Sandcat │              │ Ragdoll │
    └─────────┘              └─────────┘
```

#### Conceptos Clave

| Concepto | Descripción |
|----------|-------------|
| **Agent** | Proceso implantado en el objetivo. Recibe y ejecuta tareas. Ej: Sandcat (Go), Ragdoll (Python) |
| **Ability** | Técnica atómica mapeada a una táctica/técnica MITRE ATT&CK. Contiene comandos por plataforma |
| **Adversary** | Perfil que agrupa abilities en un orden lógico. Define el comportamiento del atacante |
| **Operation** | Ejecución de un adversario sobre un grupo de agentes. Registra todos los resultados |
| **Planner** | Algoritmo que decide qué abilities ejecutar y en qué orden |
| **Fact** | Dato extraído dinámicamente (hostname, usuario, IP) usado para abilities posteriores |
| **Source** | Colección de facts predefinidos que alimentan una operación |
| **Objective** | Criterio de éxito de una operación (número de facts recopilados) |

#### Relación con MITRE ATT&CK

Caldera organiza sus abilities siguiendo la taxonomía ATT&CK:
- **Tácticas**: categorías de alto nivel (reconnaissance, execution, persistence…)
- **Técnicas**: métodos específicos (T1082 System Info, T1059 Command Interpreter…)
- **Sub-técnicas**: variantes de una técnica (T1059.001 PowerShell, T1059.004 Bash…)

#### Casos de Uso
1. 🔴 **Red Team**: automatizar campañas de ataque contra infraestructura propia
2. 🟣 **Purple Team**: Red y Blue trabajan juntos para medir capacidades de detección
3. 🎓 **Security Training**: entrenar analistas en reconocimiento de TTPs
4. 🔍 **Threat Hunting**: simular TTPs de APTs conocidos para probar defensas

#### Caldera vs Metasploit

| Característica | Caldera | Metasploit |
|----------------|---------|------------|
| Enfoque | ATT&CK emulation | Exploit/payload |
| Automatización | Alta (planners) | Manual/semi-auto |
| Reporting | Integrado | Externo |
| Cobertura ATT&CK | Nativa | Parcial |
| Curva aprendizaje | Media | Alta |

## Sección A: Fundamentos de Caldera 🔧

### A.1 – Exploración de Abilities

In [ ]:
# ── A.1 Obtener y analizar abilities disponibles ────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/abilities', headers=HEADERS, timeout=15)
    r.raise_for_status()
    abilities = r.json()
    print(f'📋 Total de abilities disponibles: {len(abilities)}')
except Exception as e:
    abilities = []
    print(f'❌ Error al obtener abilities: {e}')

# Agrupar por táctica
tactic_counts = Counter(a.get('tactic', 'unknown') for a in abilities)
print('\n── Distribución por Táctica MITRE ATT&CK ──')
for tactic, count in sorted(tactic_counts.items(), key=lambda x: -x[1]):
    bar = '█' * min(count, 35)
    print(f'  {tactic:30s} {bar} ({count})')

In [ ]:
# ── A.1 Gráfico de barras: abilities por táctica ────────────────────────────
if tactic_counts:
    tactics_sorted = sorted(tactic_counts.items(), key=lambda x: x[1])
    labels  = [t[0] for t in tactics_sorted]
    values  = [t[1] for t in tactics_sorted]
    colors  = plt.cm.RdYlGn_r([v / max(values) for v in values])

    fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.45)))
    bars = ax.barh(labels, values, color=colors, edgecolor='black', linewidth=0.5)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_xlabel('Número de Abilities')
    ax.set_title('🔴 Abilities por Táctica MITRE ATT&CK', fontsize=13, fontweight='bold')
    ax.set_xlim(0, max(values) * 1.15)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('abilities_por_tactica.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('💾 Gráfico guardado: abilities_por_tactica.png')
else:
    print('⚠️  No hay datos de abilities para graficar')

### A.2 – Adversarios Predefinidos

Caldera incluye adversarios predefinidos que modelan comportamientos de APTs reales.
Cada adversario agrupa abilities ordenadas según la cadena de ataque del grupo emulado.

In [ ]:
# ── A.2 Listar adversarios predefinidos ─────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/adversaries', headers=HEADERS, timeout=10)
    r.raise_for_status()
    adversaries_all = r.json()
    print(f'🎭 Total adversarios: {len(adversaries_all)}')
except Exception as e:
    adversaries_all = []
    print(f'❌ Error: {e}')

print(f'\n{"Nombre":40s} {"Abilities":>9s}  Descripción')
print('-'*90)
for adv in adversaries_all:
    name   = adv.get('name', 'N/A')
    n_ab   = len(adv.get('atomic_ordering', []))
    desc   = (adv.get('description', '') or '')[:45]
    print(f'  {name:38s} {n_ab:6d}    {desc}')

In [ ]:
# ── A.2 Mapeo de técnicas por adversario ────────────────────────────────────
# Construir índice de abilities por ID
ab_index = {a['ability_id']: a for a in abilities if 'ability_id' in a}

print('── Técnicas ATT&CK por adversario (primeros 5) ──\n')
for adv in adversaries_all[:5]:
    name    = adv.get('name', 'N/A')
    ab_ids  = adv.get('atomic_ordering', [])
    print(f'🎭 {name}')
    for ab_id in ab_ids[:6]:
        ab   = ab_index.get(ab_id, {})
        tid  = ab.get('technique_id', '?')
        tname = ab.get('name', ab_id)[:50]
        tact  = ab.get('tactic', '?')
        print(f'   [{tid}] {tname} ({tact})')
    if len(ab_ids) > 6:
        print(f'   ... y {len(ab_ids)-6} más')
    print()

### A.3 – Planificadores

El planificador determina el orden y la lógica de ejecución de los abilities dentro de una operación.

| Planificador | Estrategia | Caso de uso |
|--------------|------------|-------------|
| **sequential** | Ejecuta abilities en orden definido, uno por uno | Campañas predecibles, demos |
| **batch** | Ejecuta todos los abilities disponibles a la vez | Máxima cobertura rápida |
| **landmine** | Espera a que el agente ejecute abilities espontáneamente | Detección de comportamiento |
| **likely** | Prioriza abilities con mayor probabilidad de éxito | Operaciones inteligentes |

In [ ]:
# ── A.3 Listar planificadores disponibles ────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/planners', headers=HEADERS, timeout=10)
    r.raise_for_status()
    planners = r.json()
    print(f'📋 Planificadores disponibles: {len(planners)}')
    print()
    for p in planners:
        name  = p.get('name', 'N/A')
        desc  = (p.get('description', '') or '')[:80]
        print(f'  🗂️  {name}')
        print(f'      {desc}')
        print()
except Exception as e:
    planners = []
    print(f'❌ Error al obtener planificadores: {e}')

## Sección B: Primera Operación – Reconocimiento Simple 🔍

### B.1 – Registro de Agente

Para ejecutar operaciones necesitamos al menos un agente registrado.

**Cómo desplegar un agente Sandcat (Linux):**
```bash
# En el terminal del sistema objetivo (agente):
curl -s -X POST -H 'file:sandcat.go-linux' \
     -H 'platform:linux' \
     http://localhost:8888/file/download > /tmp/sandcat
chmod +x /tmp/sandcat
/tmp/sandcat -server http://localhost:8888 -group red -v
```
> El agente se registra automáticamente en Caldera y aparece en la lista de agentes.

In [ ]:
# ── B.1 Obtener agentes disponibles ─────────────────────────────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/agents', headers=HEADERS, timeout=10)
    r.raise_for_status()
    agents_list = r.json()
    print(f'🤖 Agentes registrados: {len(agents_list)}')
except Exception as e:
    agents_list = []
    print(f'❌ Error al obtener agentes: {e}')

# Guardar primer agente para uso posterior
first_paw   = None
first_group = 'red'

if agents_list:
    print(f'\n{"PAW":12s} {"Grupo":10s} {"Host":20s} {"Plataforma":12s} {"Trusted":>8s}')
    print('-'*70)
    for ag in agents_list:
        paw      = ag.get('paw', 'N/A')
        group    = ag.get('group', 'N/A')
        host     = ag.get('host', 'N/A')
        platform = ag.get('platform', 'N/A')
        trusted  = ag.get('trusted', False)
        print(f'  {paw:10s} {group:10s} {host:20s} {platform:12s} {str(trusted):>8s}')
        if first_paw is None:
            first_paw   = paw
            first_group = group
    print(f'\n✅ Agente seleccionado: PAW={first_paw}, grupo={first_group}')
else:
    print('⚠️  Sin agentes. Las operaciones usarán grupo "red" por defecto.')

### B.2 – Crear Adversario Personalizado (Discovery)

Seleccionamos abilities de reconocimiento para crear nuestro primer adversario personalizado.

**Técnicas incluidas:**
| ID | Técnica | Descripción |
|----|---------|-------------|
| T1082 | System Information Discovery | Información del SO y hardware |
| T1057 | Process Discovery | Lista de procesos en ejecución |
| T1033 | System Owner/User Discovery | Identificar usuario actual |
| T1007 | System Service Discovery | Servicios activos en el sistema |
| T1016 | System Network Configuration | Configuración de red |

In [ ]:
# ── B.2 Filtrar abilities de discovery ───────────────────────────────────────
discovery_tactics = {'discovery', 'reconnaissance'}
discovery_ab = [a for a in abilities if a.get('tactic', '') in discovery_tactics]
print(f'🔍 Abilities de reconocimiento disponibles: {len(discovery_ab)}')

# Mostrar las primeras abilities con sus técnicas
print(f'\n{"ID Técnica":12s} {"Nombre":40s} {"Táctica":20s}')
print('-'*75)
selected_disc_ids = []
target_techniques = {'T1082', 'T1057', 'T1033', 'T1007', 'T1016'}
for ab in discovery_ab:
    tid  = ab.get('technique_id', '')
    name = ab.get('name', 'N/A')[:38]
    tact = ab.get('tactic', 'N/A')
    abid = ab.get('ability_id', '')
    marker = ' ✅' if tid in target_techniques and abid not in selected_disc_ids else ''
    if tid in target_techniques and abid not in selected_disc_ids:
        selected_disc_ids.append(abid)
    print(f'  {tid:10s} {name:40s} {tact:20s}{marker}')

# Si no tenemos suficientes, añadir las primeras disponibles
for ab in discovery_ab:
    if len(selected_disc_ids) >= 5:
        break
    abid = ab.get('ability_id', '')
    if abid not in selected_disc_ids:
        selected_disc_ids.append(abid)

print(f'\n✅ Abilities seleccionados: {len(selected_disc_ids)}')

In [ ]:
# ── B.2 Crear adversario 'Lab-Discovery' ────────────────────────────────────
adversary_disc = {
    'name': 'Lab-Discovery',
    'description': 'Adversario educativo: reconocimiento del sistema (Sección B)',
    'atomic_ordering': selected_disc_ids[:5]
}

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_disc,
                      timeout=10)
    if r.status_code in (200, 201):
        adv_disc = r.json()
        adv_disc_id = adv_disc.get('adversary_id', adv_disc.get('id', ''))
        print(f'✅ Adversario creado: {adv_disc.get("name")} (ID: {adv_disc_id})')
    else:
        print(f'⚠️  Respuesta inesperada: HTTP {r.status_code}')
        print(r.text[:200])
        adv_disc_id = None
except Exception as e:
    adv_disc_id = None
    print(f'❌ Error al crear adversario: {e}')

### B.3 – Ejecutar Operación

In [ ]:
# ── B.3 Crear y lanzar operación de reconocimiento ──────────────────────────
op_disc_id = None

if adv_disc_id:
    operation_disc = {
        'name': f'Lab-Op-Discovery-{datetime.now().strftime("%H%M%S")}',
        'adversary': {'adversary_id': adv_disc_id},
        'planner':   {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
        'group':     first_group,
        'auto_close': True,
        'state':     'running'
    }
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json=operation_disc,
                          timeout=10)
        if r.status_code in (200, 201):
            op_disc = r.json()
            op_disc_id = op_disc.get('id', '')
            print(f'✅ Operación iniciada: {op_disc.get("name")} (ID: {op_disc_id})')
            print(f'   Estado inicial: {op_disc.get("state", "N/A")}')
        else:
            print(f'⚠️  Error al crear operación: HTTP {r.status_code}')
            print(r.text[:300])
    except Exception as e:
        print(f'❌ Error: {e}')
else:
    print('⚠️  No se puede lanzar operación sin adversario válido')

In [ ]:
# ── B.3 Monitorear progreso de la operación ──────────────────────────────────
def poll_operation(op_id, max_iter=30, sleep_sec=5):
    """Sondea el estado de una operación hasta que finalice o se alcance max_iter."""
    if not op_id:
        print('⚠️  ID de operación no disponible')
        return None
    print(f'⏳ Monitoreando operación {op_id}...')
    for i in range(max_iter):
        try:
            r = requests.get(f'{CALDERA_URL}/api/v2/operations/{op_id}',
                             headers=HEADERS, timeout=10)
            if r.status_code == 200:
                op = r.json()
                state  = op.get('state', 'unknown')
                chains = op.get('chain', [])
                done   = sum(1 for c in chains if c.get('finish'))
                total  = len(chains)
                print(f'  [{i+1:2d}/{max_iter}] Estado: {state:12s} | Pasos: {done}/{total}')
                if state in ('finished', 'complete', 'cleanup'):
                    print(f'✅ Operación finalizada: {state}')
                    return op
            else:
                print(f'  ⚠️  HTTP {r.status_code}')
        except Exception as e:
            print(f'  ❌ Error de sondeo: {e}')
        time.sleep(sleep_sec)
    print('⚠️  Tiempo máximo de espera alcanzado')
    return None

op_disc_result = poll_operation(op_disc_id)

### B.4 – Analizar Resultados

In [ ]:
# ── B.4 Extraer y mostrar resultados de la operación ────────────────────────
def analyze_operation(op_data):
    """Analiza los resultados de una operación y muestra el resumen."""
    if not op_data:
        print('⚠️  Sin datos de operación para analizar')
        return []
    name   = op_data.get('name', 'N/A')
    state  = op_data.get('state', 'N/A')
    chains = op_data.get('chain', [])
    print(f'📊 Análisis de Operación: {name}')
    print(f'   Estado final: {state}')
    print(f'   Total de pasos: {len(chains)}')
    print()
    results = []
    for link in chains:
        ability  = link.get('ability', {})
        ab_name  = ability.get('name', 'N/A')[:45]
        tid      = ability.get('technique_id', 'N/A')
        tactic   = ability.get('tactic', 'N/A')
        status   = link.get('status', -1)
        output   = link.get('output', '') or ''
        icon     = '✅' if status == 0 else ('⏭️' if status == -3 else '❌')
        status_s = 'OK' if status == 0 else ('SKIP' if status == -3 else f'ERR({status})')
        print(f'  {icon} [{tid}] {ab_name}')
        print(f'      Táctica: {tactic} | Estado: {status_s}')
        if output:
            print(f'      Salida: {output[:120].strip()}')
        print()
        results.append({'ability': ab_name, 'technique_id': tid,
                        'tactic': tactic, 'status': status, 'output': output})
    return results

disc_results = analyze_operation(op_disc_result)

## Sección C: Operación Intermedia – Multi-Táctica 🎯

### C.1 – Diseño de Cadena de Ataque

Una cadena de ataque real sigue una secuencia lógica donde cada fase habilita la siguiente:

```
Discovery → Credential Access → Execution → Collection
   T1082        T1003              T1059        T1005
   T1057        T1003.001          T1059.004    T1074
   T1033
```

**¿Por qué este orden?**
1. **Discovery** primero: necesitamos conocer el entorno antes de actuar
2. **Credential Access**: obtener credenciales amplía el acceso
3. **Execution**: ejecutar payloads usando las credenciales obtenidas
4. **Collection**: recopilar datos valiosos para exfiltración posterior

**Tácticas MITRE ATT&CK involucradas:**
- `TA0007` Discovery – Conocer el entorno
- `TA0006` Credential Access – Obtener credenciales
- `TA0002` Execution – Ejecutar código
- `TA0009` Collection – Recopilar información

In [ ]:
# ── C.2 Seleccionar abilities multi-táctica ──────────────────────────────────
multi_tactics = {'discovery', 'credential-access', 'execution', 'collection'}
target_ids_c  = {'T1057', 'T1083', 'T1005', 'T1003', 'T1059', 'T1074'}

multi_ab      = [a for a in abilities if a.get('tactic', '') in multi_tactics]
print(f'🎯 Abilities multi-táctica disponibles: {len(multi_ab)}')

# Seleccionar por técnica target o los primeros disponibles por táctica
selected_multi = []
seen_tactics   = Counter()
for ab in multi_ab:
    tid   = ab.get('technique_id', '')
    tact  = ab.get('tactic', '')
    abid  = ab.get('ability_id', '')
    if abid in selected_multi:
        continue
    if tid in target_ids_c or seen_tactics[tact] < 2:
        selected_multi.append(abid)
        seen_tactics[tact] += 1
    if len(selected_multi) >= 8:
        break

# Mostrar selección
print(f'\n✅ Abilities seleccionados: {len(selected_multi)}')
print(f'\n{"Técnica":12s} {"Nombre":40s} {"Táctica":25s}')
print('-'*80)
for abid in selected_multi:
    ab   = ab_index.get(abid, {})
    tid  = ab.get('technique_id', '?')
    name = ab.get('name', abid)[:38]
    tact = ab.get('tactic', '?')
    print(f'  {tid:10s} {name:40s} {tact}')

In [ ]:
# ── C.2 Crear adversario 'Lab-MultiTactic' ───────────────────────────────────
adv_multi_id = None
adversary_multi = {
    'name': 'Lab-MultiTactic',
    'description': 'Adversario educativo: cadena multi-táctica Discovery→CredAccess→Exec→Collect',
    'atomic_ordering': selected_multi
}

try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json=adversary_multi,
                      timeout=10)
    if r.status_code in (200, 201):
        adv_m = r.json()
        adv_multi_id = adv_m.get('adversary_id', adv_m.get('id', ''))
        print(f'✅ Adversario creado: {adv_m.get("name")} (ID: {adv_multi_id})')
    else:
        print(f'⚠️  HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'❌ Error: {e}')

In [ ]:
# ── C.3 Ejecutar y monitorear operación multi-táctica ────────────────────────
op_multi_id = None
ts_start    = datetime.now()

if adv_multi_id:
    operation_multi = {
        'name': f'Lab-Op-MultiTactic-{ts_start.strftime("%H%M%S")}',
        'adversary': {'adversary_id': adv_multi_id},
        'planner':   {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
        'group':     first_group,
        'auto_close': True,
        'state':     'running'
    }
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json=operation_multi,
                          timeout=10)
        if r.status_code in (200, 201):
            op_multi_data = r.json()
            op_multi_id   = op_multi_data.get('id', '')
            print(f'✅ Operación iniciada: {op_multi_data.get("name")} (ID: {op_multi_id})')
    except Exception as e:
        print(f'❌ Error: {e}')

op_multi_result = poll_operation(op_multi_id)
ts_end = datetime.now()
print(f'\n⏱️  Duración total: {(ts_end - ts_start).seconds}s')

In [ ]:
# ── C.4 Análisis forense de la operación multi-táctica ───────────────────────
multi_results = analyze_operation(op_multi_result)

if multi_results:
    df_multi = pd.DataFrame(multi_results)
    total    = len(df_multi)
    ok       = (df_multi['status'] == 0).sum()
    skip     = (df_multi['status'] == -3).sum()
    fail     = total - ok - skip
    rate     = ok / total * 100 if total > 0 else 0

    print('\n📊 Resumen de Ejecución:')
    print(f'   Total pasos   : {total}')
    print(f'   ✅ Exitosos   : {ok}  ({rate:.1f}%)')
    print(f'   ⏭️  Omitidos  : {skip}')
    print(f'   ❌ Fallidos   : {fail}')

    print('\n── Pasos fallidos (si los hay) ──')
    failed = df_multi[df_multi['status'] != 0]
    if len(failed) > 0:
        for _, row in failed.iterrows():
            print(f'  ❌ [{row["technique_id"]}] {row["ability"]} | status={row["status"]}')
    else:
        print('  (ninguno)')

## Sección D: Operación Avanzada – Persistencia y Movimiento Lateral 🔗

### D.1 – Establecer Persistencia

La persistencia garantiza que el atacante mantenga acceso aunque el sistema se reinicie.

**Técnicas de Persistencia:**
| Técnica | ID | Método | Detección |
|---------|-----|--------|-----------|
| Scheduled Task/Job | T1053 | Cron, at, systemd timers | auditd, /var/log/cron |
| Boot/Logon Autostart | T1547 | ~/.bashrc, /etc/rc.local | File integrity monitoring |
| Create Account | T1136 | Añadir usuario oculto | /etc/passwd, useradd logs |

**Perspectiva Blue Team – Detección:**
- Monitorizar modificaciones en archivos de arranque (`/etc/cron*`, `~/.bashrc`)
- Alertar sobre nuevas tareas cron con auditd
- IDS signatures para descarga de payloads

In [ ]:
# ── D.1 Abilities de persistencia ───────────────────────────────────────────
persist_ab = [a for a in abilities if a.get('tactic', '') == 'persistence']
print(f'🔒 Abilities de persistencia disponibles: {len(persist_ab)}')

target_persist = {'T1053', 'T1547', 'T1136', 'T1546'}
selected_persist = []
for ab in persist_ab:
    tid  = ab.get('technique_id', '')
    abid = ab.get('ability_id', '')
    if abid not in selected_persist:
        selected_persist.append(abid)
    if len(selected_persist) >= 4:
        break

print(f'\n{"Técnica":12s} {"Nombre":45s}')
print('-'*60)
for abid in selected_persist:
    ab   = ab_index.get(abid, {})
    print(f'  {ab.get("technique_id","?"):10s} {ab.get("name", abid)[:43]}')

# Crear adversario de persistencia
adv_persist_id = None
try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json={'name': 'Lab-Persistence',
                            'description': 'Técnicas de persistencia (Sección D)',
                            'atomic_ordering': selected_persist},
                      timeout=10)
    if r.status_code in (200, 201):
        adv_persist_id = r.json().get('adversary_id', r.json().get('id', ''))
        print(f'\n✅ Adversario Lab-Persistence creado (ID: {adv_persist_id})')
    else:
        print(f'⚠️  HTTP {r.status_code}')
except Exception as e:
    print(f'❌ Error: {e}')

### D.2 – Movimiento Lateral

El movimiento lateral permite al atacante expandirse a otros sistemas en la red.

**Requisitos para movimiento lateral:**
- Al menos dos agentes registrados en hosts diferentes
- Credenciales válidas (obtenidas en fase de Credential Access)
- Conectividad de red entre los hosts

**Técnicas:**
| Técnica | ID | Descripción |
|---------|-----|-------------|
| Remote Services – SSH | T1021.004 | Conectar vía SSH con credenciales robadas |
| Lateral Tool Transfer | T1570 | Copiar herramientas al sistema destino |
| Remote Services | T1021 | RDP, SMB, WinRM |

In [ ]:
# ── D.2 Abilities de movimiento lateral ─────────────────────────────────────
lateral_ab = [a for a in abilities if a.get('tactic', '') == 'lateral-movement']
print(f'🔀 Abilities de movimiento lateral: {len(lateral_ab)}')

selected_lateral = []
for ab in lateral_ab[:5]:
    abid = ab.get('ability_id', '')
    name = ab.get('name', 'N/A')[:50]
    tid  = ab.get('technique_id', '?')
    print(f'  [{tid}] {name}')
    selected_lateral.append(abid)

# Crear adversario (aunque quizás no haya agentes en múltiples hosts)
adv_lateral_id = None
if selected_lateral:
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                          headers=HEADERS,
                          json={'name': 'Lab-LateralMovement',
                                'description': 'Movimiento lateral (requiere múltiples agentes)',
                                'atomic_ordering': selected_lateral[:4]},
                          timeout=10)
        if r.status_code in (200, 201):
            adv_lateral_id = r.json().get('adversary_id', r.json().get('id', ''))
            print(f'\n✅ Adversario Lab-LateralMovement creado (ID: {adv_lateral_id})')
    except Exception as e:
        print(f'❌ Error: {e}')
else:
    print('⚠️  No hay abilities de movimiento lateral disponibles')

### D.3 – Exfiltración de Datos

La exfiltración es la fase final donde los datos robados salen del entorno comprometido.

**Técnicas:**
| Técnica | ID | Canal |
|---------|-----|-------|
| Exfiltration Over C2 | T1041 | Canal C2 HTTP existente |
| Automated Exfiltration | T1020 | Scripts automáticos |
| Data Staged | T1074 | Agregación antes de exfiltrar |

In [ ]:
# ── D.3 Abilities de exfiltración y colección ───────────────────────────────
exfil_tactics = {'exfiltration', 'collection'}
exfil_ab = [a for a in abilities if a.get('tactic', '') in exfil_tactics]
print(f'📤 Abilities de exfiltración/colección: {len(exfil_ab)}')

selected_exfil = []
for ab in exfil_ab[:6]:
    abid = ab.get('ability_id', '')
    name = ab.get('name', 'N/A')[:50]
    tid  = ab.get('technique_id', '?')
    tact = ab.get('tactic', '?')
    print(f'  [{tid}] {name:50s} ({tact})')
    selected_exfil.append(abid)

adv_exfil_id = None
if selected_exfil:
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                          headers=HEADERS,
                          json={'name': 'Lab-Exfiltration',
                                'description': 'Exfiltración y colección de datos (Sección D)',
                                'atomic_ordering': selected_exfil[:5]},
                          timeout=10)
        if r.status_code in (200, 201):
            adv_exfil_id = r.json().get('adversary_id', r.json().get('id', ''))
            print(f'\n✅ Adversario Lab-Exfiltration creado (ID: {adv_exfil_id})')
    except Exception as e:
        print(f'❌ Error: {e}')

# Ejecutar operación de exfiltración
op_exfil_result = None
if adv_exfil_id:
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json={'name': f'Lab-Op-Exfil-{datetime.now().strftime("%H%M%S")}',
                               'adversary': {'adversary_id': adv_exfil_id},
                               'planner': {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
                               'group': first_group, 'auto_close': True, 'state': 'running'},
                          timeout=10)
        if r.status_code in (200, 201):
            op_exfil_id = r.json().get('id', '')
            print(f'⏳ Operación de exfiltración: {op_exfil_id}')
            op_exfil_result = poll_operation(op_exfil_id, max_iter=20)
    except Exception as e:
        print(f'❌ Error: {e}')

## Sección E: Análisis con Python/Pandas 📊

### E.1 – Extracción de Datos de Operaciones

In [ ]:
# ── E.1 Obtener todas las operaciones y construir DataFrame ──────────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS, timeout=10)
    r.raise_for_status()
    all_operations = r.json()
    print(f'📋 Total operaciones en el servidor: {len(all_operations)}')
except Exception as e:
    all_operations = []
    print(f'❌ Error: {e}')

# Construir DataFrame
ops_rows = []
for op in all_operations:
    chains   = op.get('chain', [])
    tactics  = list({c.get('ability',{}).get('tactic','?') for c in chains})
    n_ok     = sum(1 for c in chains if c.get('status') == 0)
    n_total  = len(chains)
    start    = op.get('start', '')
    finish   = op.get('finish', '')
    ops_rows.append({
        'name':        op.get('name', 'N/A'),
        'state':       op.get('state', 'N/A'),
        'group':       op.get('group', 'N/A'),
        'tactics':     ', '.join(tactics),
        'total_steps': n_total,
        'ok_steps':    n_ok,
        'fail_steps':  n_total - n_ok,
        'start':       start,
        'finish':      finish
    })

df_ops = pd.DataFrame(ops_rows)
print(f'\nDataFrame de operaciones: {df_ops.shape}')
if not df_ops.empty:
    print(df_ops.to_string(index=False))

In [ ]:
# ── E.2 Análisis de técnicas y tácticas ──────────────────────────────────────
# Aplanar todos los links de todas las operaciones
all_links = []
for op in all_operations:
    op_name = op.get('name', 'N/A')
    for link in op.get('chain', []):
        ab     = link.get('ability', {})
        status = link.get('status', -99)
        all_links.append({
            'operation':    op_name,
            'ability':      ab.get('name', 'N/A'),
            'technique_id': ab.get('technique_id', 'N/A'),
            'tactic':       ab.get('tactic', 'N/A'),
            'status':       status,
            'success':      status == 0
        })

df_links = pd.DataFrame(all_links)
print(f'Total de ejecuciones de abilities: {len(df_links)}')

if not df_links.empty:
    print('\n── Tasa de éxito por táctica ──')
    tactic_summary = df_links.groupby('tactic').agg(
        total=('success', 'count'),
        exitosos=('success', 'sum')
    ).reset_index()
    tactic_summary['tasa_%'] = (tactic_summary['exitosos'] / tactic_summary['total'] * 100).round(1)
    print(tactic_summary.to_string(index=False))

In [ ]:
# ── E.3 Visualización – 4 gráficos ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('🔴 Análisis de Operaciones Caldera', fontsize=15, fontweight='bold')

# Gráfico 1: Frecuencia de tácticas
ax1 = axes[0, 0]
if not df_links.empty:
    tc = df_links['tactic'].value_counts()
    ax1.barh(tc.index, tc.values, color='steelblue', edgecolor='black', linewidth=0.5)
    ax1.set_title('Frecuencia de Tácticas')
    ax1.set_xlabel('Ejecuciones')
    ax1.grid(axis='x', alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Frecuencia de Tácticas')

# Gráfico 2: Tasa éxito/fallo global
ax2 = axes[0, 1]
if not df_links.empty:
    total_ok   = df_links['success'].sum()
    total_fail = len(df_links) - total_ok
    if total_ok + total_fail > 0:
        ax2.pie([total_ok, total_fail],
                labels=['Exitosos', 'Fallidos/Skip'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90)
    ax2.set_title('Tasa de Éxito Global')
else:
    ax2.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Tasa de Éxito Global')

# Gráfico 3: Operaciones por estado
ax3 = axes[1, 0]
if not df_ops.empty:
    state_counts = df_ops['state'].value_counts()
    ax3.bar(state_counts.index, state_counts.values,
            color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'][:len(state_counts)],
            edgecolor='black', linewidth=0.5)
    ax3.set_title('Operaciones por Estado')
    ax3.set_ylabel('Número de operaciones')
    ax3.grid(axis='y', alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Operaciones por Estado')

# Gráfico 4: Tasa de éxito por táctica (barras)
ax4 = axes[1, 1]
if not df_links.empty and not tactic_summary.empty:
    colors_bar = ['#2ecc71' if v >= 50 else '#e74c3c' for v in tactic_summary['tasa_%']]
    ax4.bar(range(len(tactic_summary)), tactic_summary['tasa_%'],
            color=colors_bar, edgecolor='black', linewidth=0.5)
    ax4.set_xticks(range(len(tactic_summary)))
    ax4.set_xticklabels(tactic_summary['tactic'], rotation=30, ha='right', fontsize=8)
    ax4.set_title('Tasa de Éxito por Táctica (%)')
    ax4.set_ylabel('%')
    ax4.axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='50%')
    ax4.legend(fontsize=8)
    ax4.grid(axis='y', alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'Sin datos', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Tasa de Éxito por Táctica')

plt.tight_layout()
plt.savefig('caldera_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado: caldera_analysis.png')

### E.4 – Correlación Atacante → Defensor

No todas las técnicas de Caldera generan tráfico de red detectable por Suricata.
Esta tabla clasifica cada táctica según su detectabilidad:

| Táctica | Detectable por Suricata | Método de Detección | Alternativa Defensiva |
|---------|------------------------|--------------------|-----------------------|
| discovery | ❌ Local | N/A | auditd, SIEM |
| credential-access | ⚠️ Parcial | DNS queries, LDAP | auditd, PAM logs |
| execution | ⚠️ Parcial | HTTP download | EDR, auditd |
| lateral-movement | ✅ Red | SSH, SMB traffic | Suricata ET rules |
| exfiltration | ✅ Red | HTTP POST, DNS tunnel | Suricata, DLP |
| collection | ❌ Local | N/A | File integrity, auditd |
| persistence | ❌ Local | N/A | FIM, auditd |
| command-and-control | ✅ Red | Beaconing, C2 patterns | Suricata JA3/ET |

**GAP de detección:** Caldera ejecuta la mayoría de las técnicas localmente (sin tráfico de red),
lo que significa que Suricata solo detectará una fracción de las actividades.
Se necesita una solución EDR o SIEM para cobertura completa.

In [ ]:
# ── E.4 GAP analysis: técnicas vs detectabilidad ────────────────────────────
gap_data = {
    'discovery':           {'detectable': False,  'metodo': 'N/A',                  'alternativa': 'auditd, SIEM'},
    'credential-access':   {'detectable': 'parcial','metodo': 'DNS/LDAP queries',   'alternativa': 'auditd, PAM'},
    'execution':           {'detectable': 'parcial','metodo': 'HTTP download',       'alternativa': 'EDR'},
    'lateral-movement':    {'detectable': True,    'metodo': 'SSH/SMB traffic',      'alternativa': 'Suricata ET'},
    'exfiltration':        {'detectable': True,    'metodo': 'HTTP POST / DNS',      'alternativa': 'Suricata+DLP'},
    'collection':          {'detectable': False,   'metodo': 'N/A',                  'alternativa': 'FIM, auditd'},
    'persistence':         {'detectable': False,   'metodo': 'N/A',                  'alternativa': 'FIM, auditd'},
    'command-and-control': {'detectable': True,    'metodo': 'Beaconing/C2 pattern', 'alternativa': 'Suricata JA3'},
}

df_gap = pd.DataFrame.from_dict(gap_data, orient='index').reset_index()
df_gap.columns = ['Táctica', 'Detectable_Suricata', 'Método_Detección', 'Alternativa']

def det_icon(v):
    if v is True:      return '✅ Sí'
    if v is False:     return '❌ No'
    return '⚠️ Parcial'

df_gap['Detectable_Suricata'] = df_gap['Detectable_Suricata'].apply(det_icon)
print('── GAP Analysis: Técnicas Caldera vs Suricata ──\n')
print(df_gap.to_string(index=False))

## Sección F: Caso de Estudio – Campaña APT Simulada 🎭

### F.1 – Diseño de Campaña Realista

Simulamos una campaña APT completa de 8 fases, similar a grupos como APT28 o Lazarus Group:

```
FASE 1: Initial Access      ── Agente pre-implantado (simula phishing/exploit)
FASE 2: Execution           ── T1059 (Bash/sh), T1086 (PowerShell)
FASE 3: Persistence         ── T1053 (Cron), T1547 (Autostart)
FASE 4: Privilege Escalation── T1134 (Token Impersonation), T1548 (Sudo abuse)
FASE 5: Credential Access   ── T1110 (Brute Force), T1003 (OS Credential Dumping)
FASE 6: Lateral Movement    ── T1021 (Remote Services)
FASE 7: Collection          ── T1074 (Data Staged), T1123 (Audio Capture)
FASE 8: Exfiltration        ── T1041 (Exfil over C2 Channel)
```

**Duración esperada:** 10-20 minutos en un entorno de laboratorio

**Indicadores de Compromiso (IoCs) generados:**
- Conexiones HTTP regulares al C2 (beaconing)
- Comandos de enumeración del sistema
- Intentos de leer /etc/shadow o /proc/
- Archivos creados en directorios temporales

In [ ]:
# ── F.2 Crear adversario APT completo ───────────────────────────────────────
apt_tactics = ['discovery', 'execution', 'persistence', 'privilege-escalation',
               'credential-access', 'lateral-movement', 'collection', 'exfiltration']

apt_ab_selected = []
seen_tact_apt   = Counter()

for tact in apt_tactics:
    tact_abs = [a for a in abilities if a.get('tactic', '') == tact]
    for ab in tact_abs:
        abid = ab.get('ability_id', '')
        if abid not in apt_ab_selected and seen_tact_apt[tact] < 2:
            apt_ab_selected.append(abid)
            seen_tact_apt[tact] += 1
        if seen_tact_apt[tact] >= 2:
            break

print(f'🎭 Abilities seleccionados para campaña APT: {len(apt_ab_selected)}')
print('\nDistribución por táctica:')
for tact in apt_tactics:
    cnt = sum(1 for abid in apt_ab_selected
              if ab_index.get(abid, {}).get('tactic') == tact)
    bar = '█' * cnt
    print(f'  {tact:30s} {bar} ({cnt})')

adv_apt_id = None
try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json={'name': 'Lab-APTCampaign',
                            'description': 'Campaña APT simulada – 8 fases (Sección F)',
                            'atomic_ordering': apt_ab_selected},
                      timeout=10)
    if r.status_code in (200, 201):
        adv_apt_id = r.json().get('adversary_id', r.json().get('id', ''))
        print(f'\n✅ Adversario Lab-APTCampaign creado (ID: {adv_apt_id})')
    else:
        print(f'⚠️  HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'❌ Error: {e}')

In [ ]:
# ── F.2 Ejecutar campaña APT con planner batch ───────────────────────────────
op_apt_id     = None
op_apt_result = None
ts_apt_start  = datetime.now()

# Buscar ID del planner batch
batch_planner_id = 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'  # sequential por defecto
for p in planners:
    if p.get('name', '').lower() == 'batch':
        batch_planner_id = p.get('id', batch_planner_id)
        print(f'✅ Usando planner batch: {batch_planner_id}')
        break

if adv_apt_id:
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json={'name': f'Lab-APT-Campaign-{ts_apt_start.strftime("%H%M%S")}',
                               'adversary': {'adversary_id': adv_apt_id},
                               'planner': {'id': batch_planner_id},
                               'group': first_group,
                               'auto_close': True,
                               'state': 'running'},
                          timeout=10)
        if r.status_code in (200, 201):
            op_apt_data = r.json()
            op_apt_id   = op_apt_data.get('id', '')
            print(f'🎭 Campaña APT iniciada: {op_apt_data.get("name")} (ID: {op_apt_id})')
    except Exception as e:
        print(f'❌ Error: {e}')

op_apt_result = poll_operation(op_apt_id, max_iter=40, sleep_sec=7)
ts_apt_end = datetime.now()
print(f'\n⏱️  Duración campaña APT: {(ts_apt_end - ts_apt_start).seconds}s')

In [ ]:
# ── F.3 Análisis de éxito de la campaña APT ─────────────────────────────────
apt_results = analyze_operation(op_apt_result)

if apt_results:
    df_apt = pd.DataFrame(apt_results)
    total  = len(df_apt)
    ok     = (df_apt['status'] == 0).sum()
    rate   = ok / total * 100 if total > 0 else 0

    print(f'\n📊 Resumen de Campaña APT:')
    print(f'   Total técnicas ejecutadas : {total}')
    print(f'   Exitosas                  : {ok} ({rate:.1f}%)')
    print(f'   Fallidas/Omitidas         : {total - ok}')
    print(f'   Duración                  : {(ts_apt_end - ts_apt_start).seconds}s')

    # Por táctica
    print('\n── Resultados por Táctica ──')
    for tact in df_apt['tactic'].unique():
        mask  = df_apt['tactic'] == tact
        t_ok  = (df_apt[mask]['status'] == 0).sum()
        t_tot = mask.sum()
        icon  = '✅' if t_ok == t_tot else ('⚠️' if t_ok > 0 else '❌')
        print(f'  {icon} {tact:30s} {t_ok}/{t_tot}')

### F.4 – Perspectiva Blue Team

**Alertas Suricata que deberían activarse durante la campaña APT:**

| Fase | Técnica | Regla Suricata Esperada | Categoría |
|------|---------|------------------------|-----------|
| C2 Beaconing | T1071 | ET C2 Beacon detection | trojan |
| Lateral Movement | T1021 | ET LATERAL_MOVEMENT | lateral-movement |
| Exfiltración HTTP | T1041 | ET POLICY HTTP POST | policy |
| DNS Tunneling | T1071.004 | ET DNS Tunneling | dns |

**Recomendaciones IR (Incident Response):**
1. 🔍 Aislar inmediatamente el host comprometido
2. 📸 Tomar imagen forense de memoria y disco
3. 🔑 Cambiar todas las credenciales del sistema
4. 📋 Revisar todos los cron jobs y tareas de arranque
5. 🔎 Buscar persistencia adicional (backdoors, usuarios creados)

**Técnicas difíciles de detectar con IDS de red:**
- Discovery local (T1082, T1057): sin tráfico de red
- Persistence vía archivos locales: solo detectable con FIM
- Credential dumping local: requiere EDR

## Sección G: Creación de Abilities Personalizadas 🛠️

### G.1 – Estructura de un Ability

Un ability en Caldera es un objeto JSON con la siguiente estructura:

```json
{
  "id": "uuid-v4",
  "name": "Nombre descriptivo de la técnica",
  "description": "Qué hace este ability",
  "tactic": "discovery",
  "technique_id": "T1082",
  "technique_name": "System Information Discovery",
  "executors": [
    {
      "name": "sh",
      "platform": "linux",
      "command": "uname -a",
      "timeout": 30,
      "parsers": []
    }
  ]
}
```

**Campos importantes:**
| Campo | Tipo | Descripción |
|-------|------|-------------|
| `id` | UUID | Identificador único del ability |
| `tactic` | string | Táctica MITRE ATT&CK (minúsculas, con guiones) |
| `technique_id` | string | ID ATT&CK: T1082, T1059.004, etc. |
| `executors` | array | Lista de ejecutores por plataforma |
| `name` (executor) | string | sh, bash, psh (PowerShell), cmd |
| `platform` | string | linux, windows, darwin |
| `command` | string | Comando a ejecutar (base64 en algunas versiones) |
| `parsers` | array | Patrones para extraer facts del output |

In [ ]:
# ── G.2 Crear un ability personalizado ───────────────────────────────────────
import uuid as uuid_mod

custom_ability = {
    'id': str(uuid_mod.uuid4()),
    'name': 'Lab Custom - System Info Collector',
    'description': 'Recopila información del sistema para fines educativos (Lab G)',
    'tactic': 'discovery',
    'technique_id': 'T1082',
    'technique_name': 'System Information Discovery',
    'executors': [{
        'name': 'sh',
        'platform': 'linux',
        'command': ('uname -a && cat /etc/os-release 2>/dev/null | head -5 '
                   '&& echo "CPU:" && nproc && echo "RAM:" && free -h | head -2'),
        'timeout': 30
    }]
}

print('📝 Ability personalizado a crear:')
print(json.dumps(custom_ability, indent=2))

In [ ]:
# ── G.2 Enviar ability personalizado a Caldera ───────────────────────────────
custom_ab_id = None
try:
    r = requests.post(f'{CALDERA_URL}/api/v2/abilities',
                      headers=HEADERS,
                      json=custom_ability,
                      timeout=10)
    print(f'HTTP {r.status_code}')
    if r.status_code in (200, 201):
        created_ab = r.json()
        custom_ab_id = created_ab.get('ability_id', custom_ability['id'])
        print(f'✅ Ability creado: {created_ab.get("name", "N/A")}')
        print(f'   ID: {custom_ab_id}')
    else:
        # Algunos endpoints usan el ID que enviamos
        custom_ab_id = custom_ability['id']
        print(f'⚠️  Respuesta: {r.text[:300]}')
        print(f'   Usando ID proporcionado: {custom_ab_id}')
except Exception as e:
    print(f'❌ Error: {e}')
    custom_ab_id = custom_ability['id']

In [ ]:
# ── G.3 Crear adversario y operar con el ability personalizado ───────────────
adv_custom_id = None
try:
    r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                      headers=HEADERS,
                      json={'name': 'Lab-CustomAbility',
                            'description': 'Prueba de ability personalizado (Sección G)',
                            'atomic_ordering': [custom_ab_id]},
                      timeout=10)
    if r.status_code in (200, 201):
        adv_custom_id = r.json().get('adversary_id', r.json().get('id', ''))
        print(f'✅ Adversario Lab-CustomAbility (ID: {adv_custom_id})')
    else:
        print(f'⚠️  HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'❌ Error: {e}')

op_custom_result = None
if adv_custom_id:
    try:
        r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                          headers=HEADERS,
                          json={'name': f'Lab-CustomOp-{datetime.now().strftime("%H%M%S")}',
                               'adversary': {'adversary_id': adv_custom_id},
                               'planner': {'id': 'aaa7c857-37a0-4c4a-85f7-4e9f7f30e31a'},
                               'group': first_group,
                               'auto_close': True,
                               'state': 'running'},
                          timeout=10)
        if r.status_code in (200, 201):
            op_custom_id = r.json().get('id', '')
            print(f'⏳ Ejecutando operación con ability personalizado...')
            op_custom_result = poll_operation(op_custom_id, max_iter=15)
    except Exception as e:
        print(f'❌ Error: {e}')

if op_custom_result:
    for link in op_custom_result.get('chain', []):
        status = link.get('status', -99)
        output = link.get('output', '') or ''
        icon   = '✅' if status == 0 else '❌'
        print(f'{icon} Ability ejecutado | status={status}')
        if output:
            print(f'   Output:\n{output[:500]}')

## Sección H: Análisis de Evasión y Defensas 🛡️

### H.1 – Técnicas Evasivas

**¿Qué hace detectable a Caldera?**

1. **Patrón de beaconing C2**: El agente hace peticiones HTTP regulares al servidor Caldera.
   Un IDS puede detectar el intervalo regular de conexiones (ej. cada 60s).

2. **HTTP User-Agent conocido**: Sandcat usa un User-Agent por defecto reconocible.

3. **Puerto conocido**: Por defecto el servidor corre en 8888, inusual para tráfico legítimo.

4. **Payloads en texto claro**: Sin cifrado adicional, los comandos viajan en HTTP.

**Técnicas de evasión:**
| Técnica | Implementación | Efectividad |
|---------|---------------|-------------|
| Jitter en beaconing | Añadir variación aleatoria al intervalo | Alta |
| HTTPS C2 | Usar certificado TLS en Caldera | Alta |
| Domain Fronting | Redirigir C2 por CDN legítimo | Muy alta |
| Steganografía | Ocultar datos en imágenes | Media |
| Timing-based evasion | Operar solo en horario laboral | Alta |

**Regla Suricata para detectar Caldera:**
```
alert http any any -> any 8888 (
    msg:"CALDERA C2 Beaconing Detected";
    flow:established,to_server;
    http.method; content:"GET";
    http.uri; content:"/beacon";
    threshold:type both, track by_src, count 3, seconds 300;
    classtype:trojan-activity;
    sid:9000001; rev:1;
)
```

In [ ]:
# ── H.2 Consultar configuración del agente Caldera ──────────────────────────
print('── Configuración del servidor Caldera ──\n')
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/config/main', headers=HEADERS, timeout=10)
    if r.status_code == 200:
        config = r.json()
        # Mostrar parámetros relevantes (sin credenciales)
        safe_keys = ['app.name', 'app.contact.http', 'app.contact.dns',
                     'agents.implant_name', 'agents.sleep_min', 'agents.sleep_max',
                     'agents.untrusted_timer', 'app.frontend.api_base_url']
        for key in safe_keys:
            val = config.get(key, 'N/A')
            print(f'  {key:40s}: {val}')
    else:
        print(f'⚠️  HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'❌ Error: {e}')

In [ ]:
# ── H.2 Comparativa operación 'ruidosa' vs 'sigilosa' ───────────────────────
print('── Comparativa: Operación Ruidosa vs Sigilosa ──\n')

noisy_config = {
    'name': 'Op-Ruidosa',
    'description': 'Máxima velocidad, sin delays',
    'planner': 'batch',
    'sleep_min': 0,
    'sleep_max': 0,
    'jitter': 0,
    'obfuscator': 'plain-text',
}

stealthy_config = {
    'name': 'Op-Sigilosa',
    'description': 'Delays largos, jitter alto, obfuscación',
    'planner': 'likely',
    'sleep_min': 30,
    'sleep_max': 90,
    'jitter': 4,
    'obfuscator': 'base64',
}

print(f'{"Parámetro":25s} {"Ruidosa":20s} {"Sigilosa":20s}')
print('-'*68)
for k in noisy_config:
    print(f'  {k:23s} {str(noisy_config[k]):20s} {str(stealthy_config[k]):20s}')

print('\n💡 Recomendación: Usar sleep_min≥30, sleep_max≥90 y obfuscator=base64')
print('   para reducir la detectabilidad por patrones de beaconing.')

### H.3 – Defensa contra Técnicas

**Tabla de detección por táctica:**

| Táctica | Regla Suricata | Detección en Proceso | EDR/SIEM |
|---------|---------------|---------------------|----------|
| command-and-control | ET C2, JA3 fingerprint | Conexiones salientes | Sí |
| lateral-movement | ET LATERAL_MOVEMENT | SSH/SMB events | Sí |
| exfiltration | HTTP POST grande, DNS TXT | N/A | Sí (DLP) |
| execution | N/A | fork/exec audit | Sí |
| persistence | N/A | inotify on cron dirs | Sí (FIM) |

**EDR vs IDS:**
- **IDS (Suricata)**: ve solo el tráfico de RED. Invisible a actividades locales.
- **EDR**: ve syscalls, procesos, archivos. Detecta técnicas locales pero más caro.
- **SIEM**: correlaciona ambos para visibilidad completa.

**Hardening recomendado:**
```bash
# Bloquear outbound en puerto 8888 (C2 Caldera por defecto)
iptables -A OUTPUT -p tcp --dport 8888 -j DROP

# Monitorizar cron con auditd
auditctl -w /etc/cron.d/ -p wa -k cron_changes

# Restringir /proc lectura
mount -o remount,hidepid=2 /proc
```

## Sección I: Reporte Final y Comparativa Red vs Blue 📋

### I.1 – Documentar Operaciones

In [ ]:
# ── I.1 Resumen final de todas las operaciones del laboratorio ───────────────
try:
    r = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS, timeout=10)
    r.raise_for_status()
    final_ops = r.json()
except Exception as e:
    final_ops = all_operations
    print(f'⚠️  Usando operaciones en caché: {e}')

summary_rows = []
for op in final_ops:
    chains   = op.get('chain', [])
    n_ok     = sum(1 for c in chains if c.get('status') == 0)
    n_total  = len(chains)
    tactics  = sorted({c.get('ability', {}).get('tactic', '?') for c in chains})
    techniques = sorted({c.get('ability', {}).get('technique_id', '?') for c in chains})
    summary_rows.append({
        'Operación':     op.get('name', 'N/A'),
        'Estado':        op.get('state', 'N/A'),
        'Grupo':         op.get('group', 'N/A'),
        'Tácticas':      ', '.join(tactics),
        'Técnicas':      ', '.join(techniques[:5]),
        'Total_Pasos':   n_total,
        'Exitosos':      n_ok,
        'Tasa_%':        round(n_ok/n_total*100, 1) if n_total > 0 else 0
    })

df_summary = pd.DataFrame(summary_rows)
print('📊 Resumen Final del Laboratorio')
print('='*60)
if not df_summary.empty:
    print(df_summary[['Operación','Estado','Total_Pasos','Exitosos','Tasa_%']].to_string(index=False))
else:
    print('Sin operaciones completadas')

In [ ]:
# ── I.2 Exportar datos a CSV y JSON ─────────────────────────────────────────
# Exportar operaciones a CSV
if not df_summary.empty:
    df_summary.to_csv('caldera_operations_report.csv', index=False)
    print('✅ Exportado: caldera_operations_report.csv')
else:
    print('⚠️  Sin operaciones para exportar')

# Exportar técnicas a JSON
techniques_report = {}
for op in final_ops:
    for link in op.get('chain', []):
        ab  = link.get('ability', {})
        tid = ab.get('technique_id', 'N/A')
        if tid not in techniques_report:
            techniques_report[tid] = {
                'technique_name': ab.get('technique_name', 'N/A'),
                'tactic':         ab.get('tactic', 'N/A'),
                'executions':     0,
                'successes':      0
            }
        techniques_report[tid]['executions']  += 1
        techniques_report[tid]['successes']   += 1 if link.get('status') == 0 else 0

with open('caldera_techniques_report.json', 'w') as f:
    json.dump(techniques_report, f, indent=2)
print('✅ Exportado: caldera_techniques_report.json')
print(f'   Técnicas únicas: {len(techniques_report)}')

In [ ]:
# ── I.2 Guardar gráficos como PNG ────────────────────────────────────────────
# Timeline de operaciones
fig, ax = plt.subplots(figsize=(12, 5))

if not df_summary.empty:
    ops_names  = df_summary['Operación'].str[:30]
    ops_total  = df_summary['Total_Pasos']
    ops_ok     = df_summary['Exitosos']
    x = range(len(ops_names))
    ax.bar(x, ops_total, label='Total pasos',   color='#3498db', alpha=0.7)
    ax.bar(x, ops_ok,    label='Exitosos',      color='#2ecc71', alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(ops_names, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Número de pasos')
    ax.set_title('📊 Resumen de Operaciones – Red Team Lab', fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Sin datos de operaciones',
            ha='center', va='center', transform=ax.transAxes, fontsize=14)
    ax.set_title('Resumen de Operaciones')

plt.tight_layout()
plt.savefig('caldera_operations_timeline.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Guardado: caldera_operations_timeline.png')

In [ ]:
# ── I.3 Matriz de Técnicas MITRE ATT&CK ─────────────────────────────────────
# Construir la matriz desde técnicas ejecutadas
matrix_data = []
for tid, info in techniques_report.items():
    total = info['executions']
    ok    = info['successes']
    status = 'success' if ok > 0 else 'failed'
    matrix_data.append({
        'technique_id':   tid,
        'technique_name': info['technique_name'][:35],
        'tactic':         info['tactic'],
        'executions':     total,
        'successes':      ok,
        'status':         status
    })

# Añadir técnicas del catálogo que NO se ejecutaron
executed_tids = set(techniques_report.keys())
for ab in abilities[:20]:  # muestra representativa
    tid = ab.get('technique_id', '')
    if tid and tid not in executed_tids:
        matrix_data.append({
            'technique_id':   tid,
            'technique_name': ab.get('name', '')[:35],
            'tactic':         ab.get('tactic', ''),
            'executions':     0,
            'successes':      0,
            'status':         'not_tested'
        })
        executed_tids.add(tid)

df_matrix = pd.DataFrame(matrix_data)

if not df_matrix.empty:
    tested    = (df_matrix['status'] != 'not_tested').sum()
    succeeded = (df_matrix['status'] == 'success').sum()
    total_m   = len(df_matrix)
    coverage  = tested / total_m * 100 if total_m > 0 else 0

    print(f'📊 Cobertura ATT&CK: {tested}/{total_m} técnicas probadas ({coverage:.1f}%)')
    print(f'   Exitosas: {succeeded} | Fallidas: {tested-succeeded} | Sin probar: {total_m-tested}')
    print()
    # Tabla resumen por estado
    status_map = {'success': '✅ Éxito', 'failed': '❌ Falló', 'not_tested': '⬜ No probada'}
    df_matrix['Estado'] = df_matrix['status'].map(status_map)
    print(df_matrix[['technique_id','tactic','Estado']].sort_values(['tactic','technique_id']).to_string(index=False))

In [ ]:
# ── I.3 Visualización de la matriz ATT&CK ────────────────────────────────────
if not df_matrix.empty:
    color_map = {'success': '#2ecc71', 'failed': '#e74c3c', 'not_tested': '#bdc3c7'}
    df_matrix['color'] = df_matrix['status'].map(color_map)

    tactics_order = sorted(df_matrix['tactic'].unique())
    fig, ax = plt.subplots(figsize=(14, max(6, len(tactics_order) * 1.2)))

    y_pos = 0
    yticks, ylabels = [], []
    for tact in tactics_order:
        tact_df = df_matrix[df_matrix['tactic'] == tact].reset_index(drop=True)
        for i, row in tact_df.iterrows():
            ax.barh(y_pos, 1, left=i, color=row['color'], edgecolor='white', height=0.8)
        yticks.append(y_pos)
        ylabels.append(tact)
        y_pos += 1

    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel('Técnicas (cada barra = 1 técnica)')
    ax.set_title('🗺️ Matriz ATT&CK – Cobertura del Laboratorio', fontweight='bold')

    from matplotlib.patches import Patch
    legend = [Patch(color='#2ecc71', label='Exitosa'),
              Patch(color='#e74c3c', label='Falló'),
              Patch(color='#bdc3c7', label='No probada')]
    ax.legend(handles=legend, loc='lower right')
    ax.grid(axis='x', alpha=0.2)

    plt.tight_layout()
    plt.savefig('caldera_attack_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('💾 Guardado: caldera_attack_matrix.png')

### I.4 – Insights y Recomendaciones

#### Técnicas más efectivas identificadas
Las técnicas de **Discovery** (T1082, T1057, T1033) tuvieron el mayor porcentaje de éxito
ya que no requieren privilegios especiales ni conexiones de red externas.

#### Resistencia del defensor
- Las técnicas de **Lateral Movement** fallaron por falta de múltiples agentes
- Las técnicas de **Privilege Escalation** requieren condiciones específicas del SO
- La **Exfiltración** fue limitada al no haber datos sensibles reales

#### Brechas de seguridad observadas
1. 🔴 **Sin monitorización de procesos**: Discovery local pasa desapercibido
2. 🔴 **Sin FIM (File Integrity Monitoring)**: Persistencia no detectada
3. 🟡 **Suricata solo cubre red**: ~30% de las técnicas de Caldera son detectables
4. 🟢 **C2 beaconing detectable**: HTTP en puerto 8888 es obvio

#### Mejoras recomendadas
1. Implementar **auditd** para syscall monitoring
2. Desplegar **Wazuh/OSSEC** como HIDS complementario
3. Añadir **reglas Suricata ET PRO** para mejor cobertura
4. Configurar **SIEM** (Elasticsearch+Kibana) para correlación

## Sección J: Apéndices Pedagógicos 📖

### J.1 – Glosario

| Término | Definición |
|---------|------------|
| **Agent** | Proceso implantado en el sistema objetivo. Se comunica con el servidor Caldera vía C2. Ejemplos: Sandcat (Go), Ragdoll (Python), Manx (reverse shell) |
| **Ability** | Técnica atómica mapeada a una táctica y técnica MITRE ATT&CK. Contiene comandos específicos por plataforma y SO |
| **Adversary** | Perfil que agrupa abilities en un orden lógico para emular el comportamiento de un atacante real o grupo APT |
| **Operation** | Instancia de ejecución de un adversario sobre un grupo de agentes. Registra todos los comandos, outputs y timestamps |
| **Planner** | Algoritmo que decide qué abilities ejecutar, en qué orden y bajo qué condiciones dentro de una operación |
| **Fact** | Dato extraído dinámicamente del entorno (hostname, IP, usuario, archivo). Los facts permiten que las técnicas se encadenen |
| **Source** | Conjunto predefinido de facts que se inyectan al inicio de una operación como contexto inicial |
| **Objective** | Criterio de finalización de una operación: número de facts recopilados, técnica específica ejecutada, etc. |
| **PAW** | Identificador único de un agente (Persistent Agent Wrapper). Se genera aleatoriamente al registrarse |
| **Group** | Etiqueta que agrupa agentes. Las operaciones se lanzan contra un grupo, no contra un agente individual |
| **Link** | Instancia de ejecución de un ability por un agente específico. Tiene su propio estado y output |
| **Chain** | Secuencia de links ejecutados en una operación. Representa la traza completa del ataque |
| **Tactic** | Categoría de alto nivel de ATT&CK que describe el objetivo del atacante (ej: persistence, execution) |
| **Technique** | Método específico para lograr un objetivo táctico. Identificado por T + número (ej: T1082) |
| **Sub-technique** | Variante específica de una técnica. Identificada por T + número + punto + número (ej: T1059.004) |
| **C2** | Command & Control. Canal de comunicación entre el agente y el servidor Caldera |
| **Beaconing** | Patrón de comunicación periódica del agente hacia el C2 para solicitar nuevas tareas |
| **Obfuscator** | Componente que codifica los comandos antes de enviarlos al agente (plain-text, base64, etc.) |

### J.2 – Referencias

#### Documentación Oficial
- 📚 **MITRE ATT&CK Framework**: https://attack.mitre.org
- 📖 **Caldera Documentation**: https://caldera.readthedocs.io
- 💻 **Caldera GitHub**: https://github.com/mitre/caldera
- 🗺️ **ATT&CK Navigator**: https://mitre-attack.github.io/attack-navigator/

#### Herramientas Relacionadas
- 🔍 **Suricata IDS/IPS**: https://suricata.io
- 📡 **Suricata Rules (ET)**: https://rules.emergingthreats.net
- 🐍 **Atomic Red Team**: https://github.com/redcanaryco/atomic-red-team
- 🛡️ **Wazuh SIEM**: https://wazuh.com

#### Lecturas Recomendadas
- MITRE ATT&CK Design and Philosophy (whitepaper)
- MITRE Caldera – Automated Adversary Emulation (paper)
- Adversary Emulation Plans: https://mitre-engenuity.org/cybersecurity/center-for-threat-informed-defense/adversary-emulation-library/

#### Grupos APT modelados en Caldera
- APT3, APT29 (Cozy Bear), FIN6, Sandworm Team, menuPass, OilRig

### J.3 – Troubleshooting

#### ❌ Caldera no responde
```bash
# Verificar estado del servicio
systemctl status caldera
# Reiniciar
systemctl restart caldera
# Ver logs
journalctl -u caldera -n 50
```

#### ❌ Error de autenticación API (401/403)
```python
# Verificar que la API key es correcta
HEADERS = {'KEY': 'REDADMIN123', 'Content-Type': 'application/json'}
# La key por defecto está en conf/local.yml del servidor Caldera
```

#### ⚠️ No hay agentes disponibles
```bash
# Desplegar agente Sandcat manualmente
curl -s -X POST -H 'file:sandcat.go-linux' \
     -H 'platform:linux' \
     http://localhost:8888/file/download > /tmp/sandcat_agent
chmod +x /tmp/sandcat_agent
/tmp/sandcat_agent -server http://localhost:8888 -group red &
```

#### ⚠️ Operación se queda en 'running'
```python
# Forzar parada de operación
import requests
op_id = 'TU_OP_ID'
requests.patch(f'http://localhost:8888/api/v2/operations/{op_id}',
               headers=HEADERS,
               json={'state': 'finished'})
```

#### ❌ Ability falla con status != 0
- **status=-3**: Prerequisito no cumplido (falta un fact requerido)
- **status=1**: Error en la ejecución del comando
- **status=-2**: Ability enviado pero sin respuesta del agente
- Solución: verificar que el agente está activo y tiene permisos suficientes

#### 🔄 Limpiar datos del laboratorio
```python
# Borrar operaciones creadas en este lab
ops = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS).json()
for op in ops:
    if op.get('name','').startswith('Lab-'):
        requests.delete(f'{CALDERA_URL}/api/v2/operations/{op["id"]}', headers=HEADERS)
        print(f'Eliminada: {op["name"]}')
```

### J.4 – Ejercicios Prácticos 🎓

Completa los siguientes ejercicios para consolidar lo aprendido:

---

#### 🔴 Ejercicio 1: Adversario Personalizado con 5 Técnicas
> **Objetivo**: Crear un adversario personalizado con 5 técnicas de tu elección
>
> **Pasos**:
> 1. Explorar las abilities disponibles con `GET /api/v2/abilities`
> 2. Seleccionar 5 que te parezcan interesantes de distintas tácticas
> 3. Crear el adversario con `POST /api/v2/adversaries`
> 4. Ejecutar una operación y analizar los resultados
> 5. Documentar qué detecta Suricata de esas 5 técnicas

```python
# Completa el código:
my_abilities = [...]  # tus 5 ability_ids
my_adversary = {
    'name': 'Mi-Adversario-Personalizado',
    'description': '...',
    'atomic_ordering': my_abilities
}
# Crear y ejecutar...
```

---

#### 🟣 Ejercicio 2: Sequential vs Batch Planner
> **Objetivo**: Comparar el comportamiento de los planificadores sequential y batch
>
> **Pasos**:
> 1. Crear el mismo adversario dos veces
> 2. Ejecutar Operación A con planner **sequential**
> 3. Ejecutar Operación B con planner **batch**
> 4. Comparar: tiempos de ejecución, orden de abilities, resultados
> 5. Explicar las diferencias observadas

---

#### 🔵 Ejercicio 3: Ability para Técnica No Disponible
> **Objetivo**: Crear un ability personalizado para una técnica ATT&CK no incluida en Caldera
>
> **Técnicas sugeridas**:
> - T1049 System Network Connections Discovery: `ss -tulnp`
> - T1120 Peripheral Device Discovery: `lsusb && lspci`
> - T1518 Software Discovery: `dpkg -l | head -20`
>
> Crear el ability con la estructura correcta y ejecutarlo.

---

#### 🛡️ Ejercicio 4: Reglas Suricata para Detección
> **Objetivo**: Diseñar reglas Suricata para detectar las técnicas que ejecutaste
>
> **Para cada técnica de red que ejecutaste, escribe una regla Suricata:**
```
alert <protocolo> <origen> <puerto_orig> -> <destino> <puerto_dest> (
    msg:"CALDERA [Técnica] Detected";
    <condiciones>;
    classtype:trojan-activity;
    sid:<número_único>; rev:1;
)
```
> Documenta:
> - Por qué elegiste esas condiciones
> - Qué falsos positivos podría generar
> - Cómo reducir los falsos positivos